# Embed stories

How to run?
1. **Runtime --> Change runtime type --> T4 GPU**
2. Run all cells, upload `stories.jsonl` when asked
3. The last cell downloads `embeddings.zip`.

In [1]:
!pip install -q sentence-transformers
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d51b3478-556b-d75e-0cf1-6eea559e79b8)


In [2]:
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
from google.colab import files
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 16
OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

## Load stories
Order is kept exactly as in the file, so row *i* of the embeddings is story *i*.

In [3]:
import re

# remove notes that are hidden in the stories
TRAILING_NOTE = re.compile(r"\n\s*\([^()]*\)\s*$")

# some words to find the notes within the texts
NOTE_WORDS = re.compile(
    r"verteld|e-?mail|ontvangen|gepubliceerd|internet|http|RTL|\b(1[5-9]|20)\d{2}\b",
    re.I,
)


def clean_text(text):
    """Returns the story text without the source note at the end."""
    # if the text is stored as a list join the parts in one string
    if isinstance(text, list):
        text = " ".join(str(t) for t in text)
    text = text or ""
    # search for a note at the end of the text
    match = TRAILING_NOTE.search(text)
    # remove the note if it is found
    if match and NOTE_WORDS.search(match.group()):
        text = text[: match.start()]
    return text.strip()

def as_text(value):
    """Returns a field value as a plain string"""

    # if the value is a list join them together in one string
    if isinstance(value, list):
        return " ".join(str(v) for v in value)
    return value or ""

# prompt to upload the stories file
uploaded = files.upload()

stories = []
with open("stories.jsonl", "r") as f:
    # convert from JSON to a dict
    for line in f:
        stories.append(json.loads(line))

story_ids = [s["id"] for s in stories]
texts = [clean_text(s["main_text"]) for s in stories]
print(f"Loaded {len(stories)} stories")
print(f"Unique IDs: {len(set(story_ids))}")

# Make a list with the original text of every story, before cleaning
raw = [" ".join(s["main_text"]) if isinstance(s["main_text"], list) else (s["main_text"] or "") for s in stories]
print(sum(r.strip() != t for r, t in zip(raw, texts)), "texts changed")

Saving stories.jsonl to stories.jsonl
Loaded 49884 stories
Unique IDs: 49884
4585 texts changed


In [5]:
import random

# Sanity check
# Make a list with the positions of all stories whose text changed during cleaning
changed = [i for i, (r, t) in enumerate(zip(raw, texts)) if r.strip() != t]
# pick 10 random changed stories
for i in random.sample(changed, 10):
    # print only the part that was cut off
    print(repr(raw[i].strip()[len(texts[i]):]))

'\n(vgl. Sinn. 1934: 91)'
'\n(RTL4, MOPPENTOPPERS, SEPT. - OKT. 1994)'
'\n(Sinn. 1934: 115)'
'\n(Verteld op de multiculturele vertelmiddag in Houtzaagmolen De Ster op woensdag 26 april 2000, Molenpark 3)'
'\n(Jolanda Bouman, 27 februari 1992; De vriend van de vertelster hoorde het verhaal over de waakhonden in 1991, toen hij als verslaggever voor Het Vrije Volk werkte. Het leek hem een artikel waard, maar hij slaagde er niet in om van de hondeneigenaars — kennissen van zijn ouders — een bevestiging of ontkenning los te krijgen.)'
'\n(Via email vanuit Amsterdam verzonden op 8 september 1999)'
'\n(Verteld op 4 november 1999 te Lombok, Utrecht, aan Theo Meder en Marie van Dijk)'
'\n(Gevonden op 25 maart 2002 op http://onzin.ihssites.nl/pictures)'
'\n(Op 27 november 2000 aangetroffen op internet-site; adres: http://pigiprod.webgek.com/pigipoetry/sinterklaas_gedichten.htm#moppen )'
'\n(Gepubliceerd op Internet, adres http://www.concentra.be/surfmap/moppennov.html, november 1995)'


## Embed

In [6]:
# download the model and load it onto the GPU
model = SentenceTransformer(MODEL_NAME, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH
model.half()  # fp16, faster and uses less memory

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # inner product is the cosine similarity in FAISS
).astype("float32") # convert the result to float32

print(embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/3118 [00:00<?, ?it/s]

(49884, 1024)


## Save and download

In [7]:
# save the embedding matrix as embeddings.npy
np.save(OUTPUT_DIR / "embeddings.npy", embeddings)
# make a file to store the story ids
with open(OUTPUT_DIR / "story_ids.json", "w") as f:
    json.dump(story_ids, f)
# write a small summary of the settings
with open(OUTPUT_DIR / "info.json", "w") as f:
    json.dump(
        {"model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "n_stories": len(story_ids)},
        f,
    )

with zipfile.ZipFile("embeddings.zip", "w") as z:
    for file in OUTPUT_DIR.iterdir():
        z.write(file)

files.download("embeddings.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>